In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
import math

pd.set_option("display.max_colwidth", None)

# Wilson CI
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return center - margin, center + margin


def load_csv(path):
    df = pd.read_csv(path)
    df["metadata.scores.accuracy"] = pd.to_numeric(df["metadata.scores.accuracy"], errors="coerce")
    return df

# Compute overall accuracy
def compute_accuracy(df):
    rows = []
    for scaffold, group in df.groupby("metadata.scaffold"):
        n = len(group)
        k = group["metadata.scores.accuracy"].sum()
        p = k / n
        lo, hi = wilson_ci(k, n)
        rows.append({"Scaffold": scaffold, "N": n, "Correct": int(k), "Accuracy": p, "CI_lower": lo, "CI_upper": hi})
    return pd.DataFrame(rows).sort_values("Scaffold", ascending=False).reset_index(drop=True)


def fmt_wilson(r):
    return f"{r.Accuracy:.1%}  [{r.CI_lower:.1%}, {r.CI_upper:.1%}]"


def fmt_normal(r):
    margin = 1.96 * np.sqrt(r.Accuracy * (1 - r.Accuracy) / r.N)
    return f"{r.Accuracy:.1%} ± {margin:.1%}"


def make_table(result, cols, fmt_fn):
    tbl = result.copy()
    tbl["Accuracy (95% CI)"] = result.apply(fmt_fn, axis=1)
    tbl = tbl[cols]
    tbl.index = range(1, len(tbl) + 1)
    return tbl

In [ ]:
#result = compute_accuracy(load_csv("/Users/nityanadgir/Downloads/corebench-analysis/acc_saturation/all_scaffolds_updated.csv"))
result = compute_accuracy(load_csv("/path/to/all_scaffolds_updated.csv"))

display(make_table(result.sort_values("Scaffold"), ["Scaffold", "Accuracy (95% CI)"], fmt_wilson))

,Scaffold,Accuracy (95% CI)
1,"Claude Code (anthropic/claude-opus-4-5, think=10000)","89.7% [76.4%, 95.9%]"
2,Claude Code (anthropic/claude-opus-4-6),"92.3% [79.7%, 97.3%]"
3,"Codex Agent (gpt-5, re=medium)","84.6% [70.3%, 92.8%]"
4,"Codex Agent (gpt-5.1, re=medium)","87.2% [73.3%, 94.4%]"
5,"Codex Agent (gpt-5.2, re=medium)","94.9% [83.1%, 98.6%]"
6,"Codex Agent (gpt-5.3-codex, re=medium)","97.4% [86.8%, 99.5%]"
7,"Codex Agent (gpt-5.4, re=high)","97.4% [86.8%, 99.5%]"
8,"Codex Agent (gpt-5.4, re=low)","92.3% [79.7%, 97.3%]"
9,"Codex Agent (gpt-5.4, re=medium)","94.9% [83.1%, 98.6%]"
10,"Codex Agent (gpt-5.4, re=medium, mt=1)","94.9% [83.1%, 98.6%]"


In [4]:
display(make_table(result, ["Scaffold", "Accuracy (95% CI)"], fmt_normal))

,Scaffold,Accuracy (95% CI)
1,"Open Code (openai/gpt-5.4, re=high)",84.6% ± 11.3%
2,Open Code (anthropic/claude-opus-4-6),79.5% ± 12.7%
3,"Open Code (anthropic/claude-opus-4-5, think=10000)",82.1% ± 12.0%
4,"Core Agent (gpt-5.4, steps=200)",51.3% ± 15.7%
5,"Core Agent (anthropic/claude-opus-4-6, steps=200)",97.4% ± 5.0%
6,"Core Agent (anthropic/claude-opus-4-5, steps=200)",82.1% ± 12.0%
7,"Codex Agent (gpt-5.4, re=xhigh)",97.4% ± 5.0%
8,"Codex Agent (gpt-5.4, re=medium, mt=9)",97.4% ± 5.0%
9,"Codex Agent (gpt-5.4, re=medium, mt=6)",92.3% ± 8.4%
10,"Codex Agent (gpt-5.4, re=medium, mt=3)",97.4% ± 5.0%


In [ ]:
# Out of distribution test set
result_ood = compute_accuracy(load_csv("/path/to/ood_final_accuracies.csv"))
make_table(result_ood, ["Scaffold", "Correct", "N", "Accuracy (95% CI)"], fmt_wilson)

,Scaffold,Correct,N,Accuracy (95% CI)
1,"Codex Agent (gpt-5.4, re=xhigh)",19,19,"100.0% [83.2%, 100.0%]"
2,"Codex Agent (gpt-5.4, re=medium, mt=9)",16,19,"84.2% [62.4%, 94.5%]"
3,"Codex Agent (gpt-5.4, re=medium, mt=6)",16,19,"84.2% [62.4%, 94.5%]"
4,"Codex Agent (gpt-5.4, re=medium, mt=3)",17,19,"89.5% [68.6%, 97.1%]"
5,"Codex Agent (gpt-5.4, re=medium, mt=1)",18,19,"94.7% [75.4%, 99.1%]"
6,"Codex Agent (gpt-5.4, re=medium)",17,19,"89.5% [68.6%, 97.1%]"
7,"Codex Agent (gpt-5.4, re=low)",16,19,"84.2% [62.4%, 94.5%]"
8,"Codex Agent (gpt-5.4, re=high)",17,19,"89.5% [68.6%, 97.1%]"
9,"Codex Agent (gpt-5.3-codex, re=medium)",17,19,"89.5% [68.6%, 97.1%]"
10,"Codex Agent (gpt-5.2, re=medium)",19,19,"100.0% [83.2%, 100.0%]"


In [14]:
# Calculate saturation index
def se_difference(acc_top, acc_k, n, alpha):
    se = np.sqrt(((acc_top*(1-acc_top))/(n**alpha)) + ((acc_k*(1-acc_k))/(n**alpha)))
    if se == 0:
        return 1
    else:
        return se

# Calculate statistical similarity
def statistical_similarity(acc_top, acc_k, n, alpha):
    delta = acc_top - acc_k
    difference = 1.96 * se_difference(acc_top, acc_k, n, alpha)
    return delta, difference

# Calculate score compression
def score_compression(acc_top, acc_k, n, alpha):
    return (acc_top - acc_k) / se_difference(acc_top, acc_k, n, alpha)

# Calculate saturation index
def saturation_index(acc_top, acc_k, n, alpha):
    return math.exp(-1 * (score_compression(acc_top, acc_k, n, alpha)))


In [15]:
def saturation_summary(result, label, alpha=0.5):
    ranked = result.sort_values("Accuracy", ascending=False).reset_index(drop=True)
    top = ranked.iloc[0]
    fifth = ranked.iloc[4]
    n = top["N"]

    si = saturation_index(top["Accuracy"], fifth["Accuracy"], n, alpha)
    delta, margin = statistical_similarity(top["Accuracy"], fifth["Accuracy"], n, alpha)

    print(f"=== {label} ===")
    print(f"Top scaffold Accuracy:     {top['Accuracy']:.2%}")
    print(f"Fifth scaffold Accuracy:   {fifth['Accuracy']:.2%}")
    print(f"Saturation index:          {si:.4f}")
    print(f"Delta (top - fifth):       {delta:.4f}")
    print(f"1.96 * SE (similarity CI): {margin:.4f}")
    print(f"Statistically similar:     {delta <= margin}")
    print()

saturation_summary(result, "CORE-Bench v1.1 Saturation")
saturation_summary(result_ood, "CORE-Bench OOD Saturation")

=== CORE-Bench v1.1 Saturation ===
Top scaffold Accuracy:     100.00%
Fifth scaffold Accuracy:   97.44%
Saturation index:          0.6667
Delta (top - fifth):       0.0256
1.96 * SE (similarity CI): 0.1240
Statistically similar:     True

=== CORE-Bench OOD Saturation ===
Top scaffold Accuracy:     100.00%
Fifth scaffold Accuracy:   89.47%
Saturation index:          0.4887
Delta (top - fifth):       0.1053
1.96 * SE (similarity CI): 0.2881
Statistically similar:     True

